# Variable과 자동 미분(Autograd), 그리고 WanDB 시각화

이 노트북은 PyTorch의 Tensor, 자동 미분(`autograd`), 역전파(`backward`) 개념을 이해할 수 있도록 설명합니다.  



## wandb 사용 방법 상세 설명

wandb는 딥러닝 실험 과정을 기록하고 시각화하는 도구입니다. 모델 학습 중 발생하는 손실값(loss), 정확도(accuracy), 가중치(weight), 편향(bias), 그래프, 이미지 등을 웹 대시보드에서 확인할 수 있습니다.

### 1. wandb 설치
구글 코랩 또는 로컬 주피터 노트북에서 다음 명령으로 설치할 수 있습니다.

```python
!pip install wandb
```

### 2. wandb 로그인
처음 사용할 때는 아래 명령을 실행합니다.

```python
import wandb
wandb.login()
```

실행하면 API Key 입력을 요구합니다. API Key는 wandb 웹사이트의 사용자 설정 페이지에서 확인할 수 있습니다.

### 3. 실험 시작
학습을 시작하기 전에 `wandb.init()`으로 프로젝트를 생성합니다.

```python
run = wandb.init(project="프로젝트명", name="실험명")
```

### 4. 학습값 기록
학습 반복문 안에서 `wandb.log()`를 사용하면 손실값이나 정확도를 기록할 수 있습니다.

```python
wandb.log({"loss": loss.item(), "epoch": epoch})
```

### 5. 실험 종료
학습이 끝나면 `wandb.finish()`로 실험 기록을 마무리합니다.

```python
wandb.finish()
```

### 6. 오프라인 또는 수업용 실행
수업 환경에서 로그인 없이 실행하고 싶다면 `mode="disabled"` 또는 `mode="offline"`을 사용할 수 있습니다.

- `mode="disabled"`: W&B 기록 기능을 비활성화하고 코드만 정상 실행합니다.
- `mode="offline"`: 인터넷 연결 없이 로컬에 기록한 뒤 나중에 업로드할 수 있습니다.

이 노트북에서는 수강생 실습 중 로그인 오류를 줄이기 위해 기본값을 `mode="disabled"`로 설정했습니다. 실제 wandb 대시보드에 기록하려면 `WANDB_MODE = "online"`으로 바꾸고 `wandb.login()`을 실행하면 됩니다.


## 1. 필요한 라이브러리 불러오기


In [ ]:
# PyTorch는 딥러닝 모델을 만들고 학습시키기 위한 대표적인 파이썬 라이브러리입니다.
import torch

# torch.nn은 신경망 계층, 손실 함수 등 딥러닝 모델 구성에 필요한 기능을 제공합니다.
import torch.nn as nn

# torch.optim은 SGD, Adam 같은 최적화 알고리즘을 제공합니다.
import torch.optim as optim

# sys와 subprocess는 필요한 패키지가 없을 때 노트북 안에서 설치하기 위해 사용합니다.
import sys
import subprocess

# wandb가 설치되어 있지 않은 환경에서도 노트북 실습이 중단되지 않도록 처리합니다.
try:
    # wandb는 학습 과정의 손실값, 정확도, 하이퍼파라미터 등을 기록하고 시각화하는 도구입니다.
    import wandb
except ImportError:
    try:
        # wandb가 설치되어 있지 않으면 현재 파이썬 환경에 wandb를 설치합니다.
        # 구글 코랩에서는 보통 정상 설치되며, 회사/학교망에서는 인터넷 정책에 따라 실패할 수 있습니다.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "wandb"])

        # 설치가 끝난 뒤 wandb를 다시 불러옵니다.
        import wandb
    except Exception as install_error:
        # 인터넷 연결이 없거나 패키지 설치가 막힌 환경에서는 수업 흐름이 멈추지 않도록 임시 대체 객체를 만듭니다.
        # 실제 프로젝트에서는 pip install wandb를 먼저 성공시킨 뒤 아래 대체 객체 없이 wandb를 사용하는 것이 좋습니다.
        print("wandb 설치에 실패했습니다. 온라인 시각화 대신 콘솔 출력용 대체 객체를 사용합니다.")
        print("설치 오류:", install_error)

        # _DummyConfig는 wandb.config처럼 점 표기법(config.learning_rate)으로 값을 꺼내 쓰기 위한 간단한 클래스입니다.
        class _DummyConfig(dict):
            # __getattr__은 config.learning_rate처럼 속성 형태로 딕셔너리 값을 읽을 수 있게 합니다.
            def __getattr__(self, key):
                return self[key]

        # _DummyWandb는 wandb가 없을 때도 init, log, finish 코드가 실행되도록 만든 수업용 대체 클래스입니다.
        class _DummyWandb:
            # config는 wandb.config와 비슷하게 하이퍼파라미터를 저장합니다.
            config = _DummyConfig()

            # init은 실험 시작 함수처럼 동작하며 config 값을 저장합니다.
            def init(self, project=None, name=None, mode=None, config=None):
                self.config = _DummyConfig(config or {})
                print(f"[대체 W&B] project={project}, name={name}, mode={mode}")
                return self

            # log는 W&B 대시보드 기록 대신 콘솔에 값을 간단히 출력합니다.
            def log(self, data):
                if "epoch" in data and data["epoch"] % 10 == 0:
                    print("[대체 W&B 로그]", data)

            # finish는 실험 종료 메시지를 출력합니다.
            def finish(self):
                print("[대체 W&B] 실험 기록 종료")

        # 위에서 만든 대체 객체를 wandb 이름으로 사용합니다.
        wandb = _DummyWandb()

# 재현 가능한 실험을 위해 난수 시드를 고정합니다.
# 같은 코드를 다시 실행해도 가능한 한 비슷한 초기값과 결과가 나오도록 도와줍니다.
torch.manual_seed(42)

# 현재 설치된 PyTorch 버전을 출력하여 실행 환경을 확인합니다.
print("PyTorch 버전:", torch.__version__)


## 2. Tensor와 Variable 개념 이해

예전 PyTorch에서는 `Tensor`와 `Variable`을 구분했습니다.  
그러나 PyTorch 0.4 이후부터는 `Variable` 기능이 `Tensor`에 통합되었습니다.  
따라서 최신 PyTorch에서는 `Variable`을 따로 사용하지 않고, `Tensor`에 `requires_grad=True`를 설정하여 자동 미분을 사용합니다.


### 2-1. Tensor 생성하기


In [ ]:
# torch.randn(3, 4)는 평균 0, 표준편차 1을 따르는 난수로 3행 4열 Tensor를 생성합니다.
# Tensor는 PyTorch에서 숫자 데이터를 저장하는 기본 자료구조입니다.
x_tensor = torch.randn(3, 4)

# 생성된 Tensor의 값을 출력합니다.
# 출력 결과는 실행할 때마다 달라질 수 있지만, 위에서 manual_seed를 설정했기 때문에 재현성이 높아집니다.
print(x_tensor)


### 2-2. 자동 미분이 가능한 Tensor 생성하기


In [ ]:
# requires_grad=True는 이 Tensor를 이용한 모든 연산을 PyTorch가 추적하겠다는 의미입니다.
# 즉, 나중에 backward()를 호출하면 이 Tensor에 대한 기울기(gradient)를 자동으로 계산할 수 있습니다.
x = torch.randn(3, 4, requires_grad=True)

# Tensor의 실제 값을 출력합니다.
print("x 값:")
print(x)

# requires_grad 속성을 확인합니다.
# True이면 자동 미분 대상이고, False이면 자동 미분 대상이 아닙니다.
print("requires_grad:", x.requires_grad)

# 아직 backward()를 실행하지 않았기 때문에 x.grad는 None입니다.
# grad는 손실 함수가 x에 대해 얼마나 민감하게 변하는지를 나타내는 기울기 값입니다.
print("초기 gradient:", x.grad)


### 2-3. `.data`, `.grad`, `.requires_grad` 이해하기


In [ ]:
# .detach()는 현재 Tensor 값을 계산 그래프에서 분리합니다.
# 예전 코드의 .data와 비슷하게 값을 확인하는 용도로 사용할 수 있지만, 안전성을 위해 detach() 사용을 권장합니다.
x_value_only = x.detach()

# 계산 그래프에서 분리된 Tensor 값을 출력합니다.
print("계산 그래프에서 분리된 x 값:")
print(x_value_only)

# x.grad는 현재까지 계산된 x의 기울기를 저장합니다.
# 아직 backward()를 수행하지 않았으므로 None이 출력됩니다.
print("현재 x.grad:", x.grad)

# x.requires_grad는 x가 자동 미분 대상인지 확인하는 속성입니다.
print("x.requires_grad:", x.requires_grad)


## 3. 계산 그래프와 자동 미분

PyTorch는 `requires_grad=True`인 Tensor가 연산에 사용되면 계산 과정을 그래프로 기록합니다.  
이 계산 그래프를 이용하여 `backward()` 실행 시 각 변수의 기울기를 자동 계산합니다.


### 3-1. 계산 그래프 만들기


In [ ]:
# x는 자동 미분 대상 Tensor입니다.
# requires_grad=True이므로 x가 포함된 연산은 계산 그래프에 기록됩니다.
x = torch.randn(3, 4, requires_grad=True)

# y는 x를 이용해 계산된 Tensor입니다.
# y = x^2 + 4x 이므로 x가 변하면 y도 변합니다.
y = x ** 2 + 4 * x

# z는 y를 이용해 계산된 Tensor입니다.
# z = 2y + 3 이므로 x -> y -> z로 이어지는 계산 그래프가 만들어집니다.
z = 2 * y + 3

# 각 Tensor가 자동 미분 대상인지 확인합니다.
# x뿐 아니라 x로부터 만들어진 y와 z도 계산 그래프에 연결되어 있으므로 requires_grad=True입니다.
print("x.requires_grad:", x.requires_grad)
print("y.requires_grad:", y.requires_grad)
print("z.requires_grad:", z.requires_grad)

# z의 grad_fn은 z가 어떤 연산으로 만들어졌는지 보여줍니다.
# 이 정보가 계산 그래프의 연결 관계를 나타냅니다.
print("z.grad_fn:", z.grad_fn)


### 3-2. backward()로 기울기 계산하기


In [ ]:
# z는 3행 4열 Tensor이므로 바로 backward()를 호출하려면 같은 크기의 gradient 인자가 필요합니다.
# 여기서는 z의 모든 원소를 같은 중요도로 더한다고 가정하기 위해 torch.ones_like(z)를 사용합니다.
gradient_input = torch.ones_like(z)

# backward()는 z에서 x까지 거꾸로 이동하며 기울기를 계산합니다.
# 이 과정이 딥러닝에서 말하는 역전파(backpropagation)의 핵심입니다.
z.backward(gradient_input)

# x.grad에는 z가 x에 대해 얼마나 민감하게 변하는지에 대한 기울기가 저장됩니다.
# z = 2(x^2 + 4x) + 3 = 2x^2 + 8x + 3 이므로 dz/dx = 4x + 8 입니다.
print("자동 미분으로 계산된 x.grad:")
print(x.grad)

# 이론적으로 계산한 기울기 4x + 8을 직접 계산합니다.
manual_gradient = 4 * x.detach() + 8

# 자동 미분 결과와 직접 계산 결과가 같은지 비교합니다.
print("직접 계산한 기울기 4x + 8:")
print(manual_gradient)

# torch.allclose는 두 Tensor 값이 거의 같은지 확인합니다.
# 부동소수점 계산은 아주 작은 오차가 있을 수 있으므로 완전 일치 대신 allclose를 사용합니다.
print("자동 미분 결과와 직접 계산 결과가 같은가?:", torch.allclose(x.grad, manual_gradient))


## 4. wandb를 사용한 선형 회귀 학습 시각화

아래 예제는 간단한 선형 회귀 모델을 학습하면서 손실값, 가중치, 편향을 wandb에 기록하는 코드입니다.  
wandb는 `wandb.log()`로 기록한 값을 대시보드에서 자동으로 그래프로 보여줍니다.


### 4-1. wandb 실행 모드 설정


In [ ]:
# 수업 환경에서는 W&B 로그인이 되어 있지 않을 수 있으므로 기본값을 disabled로 설정합니다.
# 실제 wandb 웹 대시보드에 기록하려면 아래 값을 "online"으로 바꾸고 wandb.login()을 먼저 실행합니다.
WANDB_MODE = "disabled"

# wandb.init()은 하나의 실험 실행(run)을 시작합니다.
# project는 wandb에서 묶어서 관리할 프로젝트 이름입니다.
# name은 현재 실험의 이름입니다.
# config에는 학습률, 반복 횟수 같은 하이퍼파라미터를 저장합니다.
run = wandb.init(
    project="pytorch-autograd-linear-regression",
    name="variable-autograd-wandb-example",
    mode=WANDB_MODE,
    config={
        "learning_rate": 0.01,
        "epochs": 100,
        "model": "Linear Regression",
        "optimizer": "SGD",
        "loss_function": "MSELoss"
    }
)

# wandb.config는 위에서 등록한 하이퍼파라미터를 코드에서 쉽게 꺼내 쓰기 위한 객체입니다.
config = wandb.config

# 현재 wandb 실행 모드를 출력합니다.
print("현재 W&B 실행 모드:", WANDB_MODE)


### 4-2. 학습 데이터 생성


In [ ]:
# torch.linspace(-5, 5, 100)는 -5부터 5까지 균등한 간격의 숫자 100개를 생성합니다.
# unsqueeze(1)은 모양을 [100]에서 [100, 1]로 바꾸어 모델 입력 형태에 맞춥니다.
X = torch.linspace(-5, 5, 100).unsqueeze(1)

# 실제 정답 관계를 y = 2x + 1로 가정합니다.
# 여기에 작은 난수 잡음을 더해 실제 데이터처럼 완전히 직선이 아니게 만듭니다.
y = 2 * X + 1 + 0.5 * torch.randn(X.size())

# 입력 데이터와 정답 데이터의 크기를 확인합니다.
print("X 크기:", X.shape)
print("y 크기:", y.shape)

# 앞쪽 데이터 5개를 확인하여 데이터 형태를 이해합니다.
print("X 앞 5개:")
print(X[:5])
print("y 앞 5개:")
print(y[:5])


### 4-3. 선형 회귀 모델, 손실 함수, 최적화 알고리즘 정의


In [ ]:
# nn.Linear(1, 1)은 입력값 1개를 받아 출력값 1개를 만드는 선형 회귀 계층입니다.
# 내부적으로 y = weight * x + bias 형태의 계산을 수행합니다.
model = nn.Linear(1, 1)

# nn.MSELoss()는 평균제곱오차 손실 함수입니다.
# 예측값과 실제값의 차이를 제곱한 뒤 평균을 내므로 회귀 문제에서 자주 사용됩니다.
criterion = nn.MSELoss()

# optim.SGD는 확률적 경사하강법 최적화 알고리즘입니다.
# model.parameters()는 학습 가능한 weight와 bias를 optimizer에 전달합니다.
# lr은 learning rate, 즉 한 번 업데이트할 때 얼마나 크게 이동할지 정하는 값입니다.
optimizer = optim.SGD(model.parameters(), lr=config.learning_rate)

# 모델의 초기 weight와 bias를 출력합니다.
# 학습 전에는 임의의 값으로 초기화되어 있습니다.
print("초기 weight:", model.weight.item())
print("초기 bias:", model.bias.item())


### 4-4. 모델 학습 및 wandb 기록


In [ ]:
# config.epochs만큼 반복 학습을 수행합니다.
# epoch는 전체 학습 데이터를 한 번 사용하여 모델을 업데이트하는 단위입니다.
for epoch in range(config.epochs):
    # optimizer.zero_grad()는 이전 반복에서 누적된 기울기를 0으로 초기화합니다.
    # PyTorch는 기본적으로 gradient를 누적하므로 매 학습 단계마다 초기화해야 합니다.
    optimizer.zero_grad()

    # model(X)는 현재 weight와 bias를 사용하여 예측값을 계산합니다.
    predictions = model(X)

    # criterion(predictions, y)는 예측값과 실제값 사이의 평균제곱오차를 계산합니다.
    loss = criterion(predictions, y)

    # loss.backward()는 손실값을 기준으로 weight와 bias의 기울기를 자동 계산합니다.
    loss.backward()

    # optimizer.step()은 계산된 기울기를 이용하여 weight와 bias를 업데이트합니다.
    optimizer.step()

    # 현재 weight 값을 파이썬 숫자로 꺼냅니다.
    current_weight = model.weight.item()

    # 현재 bias 값을 파이썬 숫자로 꺼냅니다.
    current_bias = model.bias.item()

    # wandb.log()는 학습 중 기록하고 싶은 값을 W&B에 저장합니다.
    # Visdom에서 line plot을 그리던 부분을 W&B 로그 기록 방식으로 대체한 것입니다.
    wandb.log({
        "epoch": epoch + 1,
        "loss": loss.item(),
        "weight": current_weight,
        "bias": current_bias
    })

    # 10 epoch마다 학습 진행 상황을 출력합니다.
    # 수업 중에는 출력값을 보면서 손실이 줄어드는지 확인할 수 있습니다.
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1:03d}/{config.epochs}] | Loss: {loss.item():.6f} | Weight: {current_weight:.4f} | Bias: {current_bias:.4f}")


### 4-5. 학습 결과 확인 및 wandb 종료


In [ ]:
# 학습이 끝난 뒤 최종 weight와 bias를 출력합니다.
# 데이터가 y = 2x + 1 근처로 만들어졌기 때문에 weight는 2, bias는 1에 가까워지는 것이 목표입니다.
print("최종 weight:", model.weight.item())
print("최종 bias:", model.bias.item())

# torch.no_grad()는 예측 과정에서 gradient 계산을 하지 않도록 설정합니다.
# 평가나 예측 단계에서는 역전파가 필요 없으므로 메모리와 계산량을 줄일 수 있습니다.
with torch.no_grad():
    # 학습된 모델을 사용하여 전체 X에 대한 예측값을 계산합니다.
    final_predictions = model(X)

    # 최종 손실값을 다시 계산합니다.
    final_loss = criterion(final_predictions, y)

# 최종 손실값을 출력합니다.
print("최종 손실값:", final_loss.item())

# 최종 결과도 wandb에 한 번 더 기록합니다.
wandb.log({
    "final_loss": final_loss.item(),
    "final_weight": model.weight.item(),
    "final_bias": model.bias.item()
})

# wandb.finish()는 현재 실험 기록을 종료합니다.
# online 모드에서는 대시보드 업로드가 마무리되고, disabled 모드에서는 조용히 종료됩니다.
wandb.finish()
